# 7. Final targets: six event categories

Station Operations, Speed Deviation – Maximum Authorized Speed, Emergency / Penalty Brake Application, Mechanical – Less Severe, Signal Events – On Board Control System, Speed Deviation – Temporary Speed Restriction.

### Run the earlier steps
Loads the variables this notebook needs (`df`, `df_norm`, ...) by running `06_callback_subset.ipynb` (which runs the steps before it). Output is hidden.

In [1]:
%%capture
%run -i "06_callback_subset.ipynb"

## 7.1 Event diagnostic 1: label support in each partition

In [2]:
# Event diagnostic 1: check the six outputs in the existing split preview.
# Run after the earlier diagnostics. Do not overwrite df or its partitions.

required_variables = ["df", "diag", "diag_partitions"]
missing = [name for name in required_variables if name not in globals()]
if missing:
    raise ValueError(
        f"Run the earlier target diagnostics 1-3 first. Missing: {missing}"
    )
if not isinstance(df, pd.DataFrame) or df.empty:
    raise ValueError("df must contain the full CSV.")

event_headers = [" ".join(str(c).split()) for c in df.columns]
if len(event_headers) != len(set(event_headers)):
    raise ValueError("Column names collide after whitespace normalization.")
event_lookup = dict(zip(event_headers, df.columns))
event_names = [
    "Operating Cab / Car Event - Station Operations",
    "Speed Deviation - Maximum Authorized Speed",
    "Braking Event - Emergency / Penalty Brake Application",
    "Mechanical - Less Severe",
    "Signal Events - On Board Control System",
    "Speed Deviation - Temporary Speed Restriction",
]
event_targets = ["Events Anomaly / " + name for name in event_names]
event_narrative_cols = sorted(
    [c for c in event_headers
     if re.fullmatch(r"Person Narrative / Narrative \d+", c)],
    key=lambda c: int(c.rsplit(" ", 1)[1]),
)
event_callback_cols = sorted(
    [c for c in event_headers
     if re.fullmatch(r"Person Callback / Callback \d+", c)],
    key=lambda c: int(c.rsplit(" ", 1)[1]),
)
event_keep = (
    ["ACN", "Time Date", "Synopsis"] + event_targets
    + event_narrative_cols + event_callback_cols
)
missing = [c for c in event_keep if c not in event_lookup]
if missing or not event_narrative_cols or not event_callback_cols:
    raise KeyError(f"Missing target or text columns. Missing names: {missing}")

event_check = df[[event_lookup[c] for c in event_keep]].copy()
event_check = event_check.reset_index(drop=True)
event_check.columns = event_keep
if not event_check["ACN"].astype("string").equals(
    diag["ACN"].reset_index(drop=True).astype("string")
):
    raise ValueError("The dataframe changed. Rerun the earlier diagnostics.")

event_parts = {
    name: np.asarray(diag_partitions[name])
    for name in ["Training", "Validation", "Test"]
}
all_positions = np.concatenate(list(event_parts.values()))
if not np.array_equal(np.sort(all_positions), np.arange(len(event_check))):
    raise ValueError("The existing preview does not partition these rows exactly once.")

event_values = event_check[event_targets].astype("string").fillna("").apply(
    lambda s: s.str.strip().str.lower()
)
true_values = ["true", "1", "1.0", "yes"]
allowed_values = true_values + ["false", "0", "0.0", "no", ""]
if not event_values.isin(allowed_values).all().all():
    raise ValueError("Unexpected target values: inspect before creating labels.")
# Labels represent recorded code selections, not confirmed real-world absence.
event_y = event_values.isin(true_values).astype("int64")
event_y.columns = event_names

# Reproduce the previous candidate-text screen, without editing source text.
event_narratives = event_check[event_narrative_cols].astype("string").fillna("")
event_callbacks = event_check[event_callback_cols].astype("string").fillna("")
event_placeholder = event_narratives.apply(
    lambda s: s.str.strip().str.fullmatch(
        r"\[?(?:(?:second|third) person|person \d+)(?: reference)? "
        r"added to supplement the records?\.?\]?\.?", case=False, na=False
    )
)
event_narrative_present = event_narratives.apply(lambda s: s.str.strip().ne(""))
event_callback_candidate = event_callbacks.apply(
    lambda s: s.str.strip().ne("")
    & ~s.str.strip().str.lower().str.startswith("unable to contact")
)
event_callback_mask = (
    (event_narrative_present & ~event_placeholder).any(axis=1)
    & event_callback_candidate.any(axis=1)
)

event_support = pd.DataFrame(index=event_names)
for name, positions in event_parts.items():
    paired = positions[event_callback_mask.iloc[positions].to_numpy()]
    event_support[f"{name}: selected"] = event_y.iloc[positions].sum()
    event_support[f"{name}: selected with callback"] = event_y.iloc[paired].sum()
    print(f"{name}: {len(positions):,} reports; {len(paired):,} candidate callback reports")

event_support.index.name = "Recorded category"
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(event_support)
print("Counts refer to the OLD split preview, which was stratified on Safety Concern.")
print("This checks support for the new labels; it does not finalize a new split.")
print("No source rows were removed, no predictors selected, and no model fitted.")


Training: 7,182 reports; 3,287 candidate callback reports
Validation: 1,539 reports; 706 candidate callback reports
Test: 1,540 reports; 705 candidate callback reports


,Training: selected,Training: selected with callback,Validation: selected,Validation: selected with callback,Test: selected,Test: selected with callback
Recorded category,,,,,,
Operating Cab / Car Event - Station Operations,1784,862,370,180,387,185
Speed Deviation - Maximum Authorized Speed,1448,624,286,124,308,125
Braking Event - Emergency / Penalty Brake Application,1301,607,269,125,278,118
Mechanical - Less Severe,1153,530,250,112,254,115
Signal Events - On Board Control System,991,446,228,103,211,95
Speed Deviation - Temporary Speed Restriction,950,388,210,87,213,90


Counts refer to the OLD split preview, which was stratified on Safety Concern.
This checks support for the new labels; it does not finalize a new split.
No source rows were removed, no predictors selected, and no model fitted.


## 7.2 Event diagnostic 2: qualitative review of labels, narratives and callbacks

In [3]:
# Event diagnostic 2: review narrative/callback evidence for the proposed outputs.
# Two selected reports per category, from TRAINING + candidate callbacks only.
# This is a small qualitative review, not a measure of label accuracy.

if not all(name in globals() for name in [
    "event_check", "event_y", "event_parts", "event_callback_mask"
]):
    raise ValueError("Run Event diagnostic 1 first.")

EXAMPLES_PER_CATEGORY = 2
used_positions = set()
event_sample_rows = []
review_sections = [
    "# C3RS event-label and text review\n",
    "Purpose: inspect what supports the recorded labels and what callbacks add.\n",
    "Training-preview records only; sampled among reports with candidate callbacks. "
    "This intentionally selected sample cannot estimate population label accuracy "
    "or the proportion of callbacks that are useful.\n",
    "Text is retained in full and organized by original person slot. "
    "Different people may supply the narrative and callback in the same record.\n",
]

for label_number, category in enumerate(event_names):
    pool = event_y.loc[event_parts["Training"], category]
    eligible = (
        pool.eq(1)
        & event_callback_mask.loc[pool.index]
        & ~pool.index.isin(used_positions)
    )
    choices = pool.loc[eligible]
    if choices.empty:
        print(f"No eligible unreviewed training records for: {category}")
        continue
    sampled = choices.sample(
        n=min(EXAMPLES_PER_CATEGORY, len(choices)),
        random_state=42 + label_number,
    ).index

    for position in sampled:
        used_positions.add(int(position))
        row = event_check.loc[position]
        recorded = event_y.columns[event_y.loc[position].eq(1)].tolist()
        event_sample_rows.append({
            "Category sampled": category,
            "ACN": row["ACN"],
            "Date": row["Time Date"],
            "Six study codes selected": "; ".join(recorded),
        })
        review_sections.extend([
            f"\n## Example {len(event_sample_rows)} | ACN {row['ACN']}\n",
            f"**Category sampled:** {category}\n",
            f"**Date:** {row['Time Date']}\n",
            "**Six study codes selected:** " + "; ".join(recorded) + "\n",
            "### Synopsis\n",
            str(row["Synopsis"]) if pd.notna(row["Synopsis"]) else "[Blank]",
        ])

        slots = sorted({int(c.rsplit(" ", 1)[1])
                        for c in event_narrative_cols + event_callback_cols})
        for slot in slots:
            for kind in ["Narrative", "Callback"]:
                column = f"Person {kind} / {kind} {slot}"
                if column not in row.index or pd.isna(row[column]):
                    continue
                text = str(row[column])
                if not text.strip():
                    continue
                review_sections.extend([f"\n### {kind} {slot}\n", text])

        review_sections.extend([
            "\n### Review notes (leave uncertain items unresolved)\n",
            "- Narrative evidence for the sampled category: [quote / unclear]\n",
            "- Callback: [adds event detail / adds context only / repeats / unclear]\n",
            "- Synopsis: [describes the category / omits it / unclear]\n",
            "- Disagreement, missing detail, or label-definition question: [notes]\n",
        ])

event_sample_table = pd.DataFrame(event_sample_rows)
with pd.option_context("display.max_colwidth", None):
    display(event_sample_table)

event_review_path = Path("C3RS_event_label_review.md")
event_review_path.write_text("\n\n".join(review_sections), encoding="utf-8")
print(f"Saved {len(event_sample_table)} full report examples to: {event_review_path.resolve()}")
print("Open or share that Markdown file; no narratives were truncated.")
print("The file contains TRAINING examples only. No labels or source data were changed.")


,Category sampled,ACN,Date,Six study codes selected
0,Operating Cab / Car Event - Station Operations,28213,2023-04,Operating Cab / Car Event - Station Operations
1,Operating Cab / Car Event - Station Operations,23538,2021-09,Operating Cab / Car Event - Station Operations
2,Speed Deviation - Maximum Authorized Speed,34021,2024-10,Speed Deviation - Maximum Authorized Speed; Braking Event - Emergency / Penalty Brake Application
3,Speed Deviation - Maximum Authorized Speed,36083,2025-03,Speed Deviation - Maximum Authorized Speed; Braking Event - Emergency / Penalty Brake Application; Speed Deviation - Temporary Speed Restriction
4,Braking Event - Emergency / Penalty Brake Application,24132,2021-11,Speed Deviation - Maximum Authorized Speed; Braking Event - Emergency / Penalty Brake Application; Signal Events - On Board Control System
5,Braking Event - Emergency / Penalty Brake Application,35264,2025-01,Speed Deviation - Maximum Authorized Speed; Braking Event - Emergency / Penalty Brake Application
6,Mechanical - Less Severe,40628,2026-02,Mechanical - Less Severe; Speed Deviation - Temporary Speed Restriction
7,Mechanical - Less Severe,31953,2024-04,Operating Cab / Car Event - Station Operations; Mechanical - Less Severe
8,Signal Events - On Board Control System,41072,2026-03,Mechanical - Less Severe; Signal Events - On Board Control System
9,Signal Events - On Board Control System,39841,2025-12,Mechanical - Less Severe; Signal Events - On Board Control System


Saved 12 full report examples to: /workspaces/C3R3-Railway-Safety-ML/src/C3RS_event_label_review.md
Open or share that Markdown file; no narratives were truncated.
The file contains TRAINING examples only. No labels or source data were changed.
